# Prescriptive Analytics Demo

This notebook demonstrates a full workflow for working with synthetic EHR data from CSV files using both SQL and pandas. We'll:

- Load CSVs
- Create SQLite in-memory tables
- Use SQL to join and filter
- Use pandas for sorting, grouping, and plotting
- Export results


In [4]:
import pandas
import sqlite3
import matplotlib.pyplot as plt
from IPython.display import display


# Specify the disk drive path to the folder containing the Synthea CSV files
# You'll change the path on the next line to match your system setup
my_path = "/Users/spb24/Downloads/coherent-11-07-2022/csv/"

# Load the CSV files
patients = pandas.read_csv(my_path + "patients.csv")
encounters = pandas.read_csv(my_path + "encounters.csv")
# Make sure the 'BIRTHDATE' column is datetime
patients['BIRTHDATE'] = pandas.to_datetime(patients['BIRTHDATE'], errors='coerce')

# Define 'today' correctly
today = pandas.Timestamp.today()

# Rename patients[id] to match key in encounters
patients['patient_id'] = patients['Id'].copy()

# Compute age in years
patients['age'] = (today - patients['BIRTHDATE']) / pandas.Timedelta(days=365.25)

# Thus far, we have created dataframes, not yet sql tables
display(patients.head())
display(encounters.head())

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,...,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,patient_id,age
0,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,1957-04-07,NaN,999-32-2242,S99944366,X13210523X,Mr.,Gregg522,Cummerata161,NaN,...,Amherst,Massachusetts,Hampshire County,NaN,42.392261,-72.479152,1318721.22,4100.44,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,68.021319
1,b8eb8d31-1031-fb5b-e207-b9815f80744c,1975-08-16,NaN,999-70-2742,S99952609,X70704838X,Mr.,Lemuel304,Rodriguez71,NaN,...,Framingham,Massachusetts,Middlesex County,1702.0,42.263233,-71.481261,9089.75,0.00,b8eb8d31-1031-fb5b-e207-b9815f80744c,49.664029
2,ce9bd436-6b59-0452-86a4-61f3642736bc,1945-05-11,2015-04-09,999-54-1330,S99979547,X86639992X,Mr.,Leonardo412,Klein929,NaN,...,New Bedford,Massachusetts,Bristol County,2744.0,41.665030,-70.914507,230547.34,5821.16,ce9bd436-6b59-0452-86a4-61f3642736bc,79.928232
3,6fc3e360-ae68-c411-e091-4734df51eb18,1947-12-30,NaN,999-59-9652,S99949959,X34069329X,Mrs.,Adelle43,Dooley940,NaN,...,Wellesley,Massachusetts,Norfolk County,2457.0,42.311830,-71.286312,1568950.96,8597.39,6fc3e360-ae68-c411-e091-4734df51eb18,77.291681
4,ce4ce4d8-d4e2-aca2-5a92-8ce703c5077a,1993-02-05,NaN,999-11-2438,S99932651,X80439009X,Mrs.,Veta780,Spencer878,NaN,...,New Bedford,Massachusetts,Bristol County,2747.0,41.671645,-70.914871,615324.99,9184.78,ce4ce4d8-d4e2-aca2-5a92-8ce703c5077a,32.188327


,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
0,03bc904a-c655-0580-a620-264b065c01e6,1991-06-23T08:57:56Z,1991-06-23T09:12:56Z,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,7185ef1a-1b68-3723-8804-32dacb99e678,5ab0650d-1376-3bde-83c3-7f517505bdb7,d47b3510-2895-3b70-9897-342d681c769d,wellness,162673000,General examination of patient (procedure),129.16,129.16,49.16,NaN,NaN
1,7cfc42ac-357a-c88e-1275-9d63f900b7d8,2001-04-22T08:57:56Z,2001-04-22T09:12:56Z,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,7185ef1a-1b68-3723-8804-32dacb99e678,5ab0650d-1376-3bde-83c3-7f517505bdb7,d47b3510-2895-3b70-9897-342d681c769d,wellness,162673000,General examination of patient (procedure),129.16,129.16,49.16,NaN,NaN
2,42dbe56f-5b30-38b3-5634-a9b1a0206e44,1993-10-09T13:11:33Z,1993-10-09T13:26:33Z,b8eb8d31-1031-fb5b-e207-b9815f80744c,87f6941d-90f6-3c1d-b443-4038458cf8d7,5db62284-9e52-3c8e-bde0-53d81bd39963,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,wellness,162673000,General examination of patient (procedure),129.16,129.16,0.00,NaN,NaN
3,d3cbf195-f49a-24f5-fe79-c6dc21fb92da,2003-04-27T08:57:56Z,2003-04-27T09:12:56Z,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,7185ef1a-1b68-3723-8804-32dacb99e678,5ab0650d-1376-3bde-83c3-7f517505bdb7,d47b3510-2895-3b70-9897-342d681c769d,wellness,162673000,General examination of patient (procedure),129.16,129.16,49.16,NaN,NaN
4,155aa73b-46da-5808-c218-80a5ed671009,2003-04-27T08:57:56Z,2003-04-27T09:12:56Z,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,49318f80-bd8b-3fc7-a096-ac43088b0c12,6649b133-96f5-3920-90c7-2e61723d7dc8,d47b3510-2895-3b70-9897-342d681c769d,ambulatory,185347001,Encounter for problem,77.49,77.49,0.00,271737000.0,Anemia (disorder)


## Create SQLite Database in Memory

We'll use `sqlite3` to create a temporary database, then load our CSVs into it as SQL tables.


In [6]:
rdbms = sqlite3.connect(":memory:")
patients.to_sql("patients", rdbms, index=False, if_exists="replace")
encounters.to_sql("encounters", rdbms, index=False, if_exists="replace")


285339

## Patients Over 50

Filter for patients over age 50 using SQL.


In [10]:
# Spell out what we intend to do
query = '''
SELECT * FROM patients WHERE (AGE > 50) and (AGE < 100);
'''

# Do it
patients_over_50 = pandas.read_sql_query(query, rdbms)
display(patients_over_50.head())

# The above syntax is the same as
patients_over_40 = pandas.read_sql_query("SELECT * FROM patients WHERE (AGE >40) and (AGE <100)", rdbms)
display(patients_over_40.head())

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,...,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,patient_id,age
0,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,1957-04-07 00:00:00,None,999-32-2242,S99944366,X13210523X,Mr.,Gregg522,Cummerata161,None,...,Amherst,Massachusetts,Hampshire County,NaN,42.392261,-72.479152,1318721.22,4100.44,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,68.021319
1,ce9bd436-6b59-0452-86a4-61f3642736bc,1945-05-11 00:00:00,2015-04-09,999-54-1330,S99979547,X86639992X,Mr.,Leonardo412,Klein929,None,...,New Bedford,Massachusetts,Bristol County,2744.0,41.665030,-70.914507,230547.34,5821.16,ce9bd436-6b59-0452-86a4-61f3642736bc,79.928232
2,6fc3e360-ae68-c411-e091-4734df51eb18,1947-12-30 00:00:00,None,999-59-9652,S99949959,X34069329X,Mrs.,Adelle43,Dooley940,None,...,Wellesley,Massachusetts,Norfolk County,2457.0,42.311830,-71.286312,1568950.96,8597.39,6fc3e360-ae68-c411-e091-4734df51eb18,77.291681
3,b4c809c0-de80-1c62-ab58-7b885e475e76,1965-11-23 00:00:00,None,999-49-5162,S99931885,X24243962X,Mrs.,Mittie376,Metz686,None,...,Haverhill,Massachusetts,Essex County,NaN,42.827782,-71.151689,1238586.78,11146.30,b4c809c0-de80-1c62-ab58-7b885e475e76,59.391613
4,4fdafa07-e5cb-9b2a-55c6-50902c49ccfb,1930-01-02 00:00:00,1937-09-22,999-80-8436,None,None,None,Dara636,Miller503,None,...,Concord,Massachusetts,Middlesex County,NaN,42.453890,-71.374585,2759.66,0.00,4fdafa07-e5cb-9b2a-55c6-50902c49ccfb,95.282099


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,...,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,patient_id,age
0,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,1957-04-07 00:00:00,None,999-32-2242,S99944366,X13210523X,Mr.,Gregg522,Cummerata161,None,...,Amherst,Massachusetts,Hampshire County,NaN,42.392261,-72.479152,1318721.22,4100.44,8b0484cd-3dbd-8b8d-1b72-a32f74a5a846,68.021319
1,b8eb8d31-1031-fb5b-e207-b9815f80744c,1975-08-16 00:00:00,None,999-70-2742,S99952609,X70704838X,Mr.,Lemuel304,Rodriguez71,None,...,Framingham,Massachusetts,Middlesex County,1702.0,42.263233,-71.481261,9089.75,0.00,b8eb8d31-1031-fb5b-e207-b9815f80744c,49.664029
2,ce9bd436-6b59-0452-86a4-61f3642736bc,1945-05-11 00:00:00,2015-04-09,999-54-1330,S99979547,X86639992X,Mr.,Leonardo412,Klein929,None,...,New Bedford,Massachusetts,Bristol County,2744.0,41.665030,-70.914507,230547.34,5821.16,ce9bd436-6b59-0452-86a4-61f3642736bc,79.928232
3,6fc3e360-ae68-c411-e091-4734df51eb18,1947-12-30 00:00:00,None,999-59-9652,S99949959,X34069329X,Mrs.,Adelle43,Dooley940,None,...,Wellesley,Massachusetts,Norfolk County,2457.0,42.311830,-71.286312,1568950.96,8597.39,6fc3e360-ae68-c411-e091-4734df51eb18,77.291681
4,b4c809c0-de80-1c62-ab58-7b885e475e76,1965-11-23 00:00:00,None,999-49-5162,S99931885,X24243962X,Mrs.,Mittie376,Metz686,None,...,Haverhill,Massachusetts,Essex County,NaN,42.827782,-71.151689,1238586.78,11146.30,b4c809c0-de80-1c62-ab58-7b885e475e76,59.391613


## Last Preventive Visit per Patient

We'll identify the last recorded encounter for each patient classified as a wellness/preventive visit.


In [12]:
# Assign the query we intend to run to a variable named 'query'
query = '''
SELECT
    encounters.PATIENT AS patient_id,
    MAX(encounters.START) AS last_preventive
FROM encounters 
WHERE encounters.ENCOUNTERCLASS LIKE '%wellness%'
GROUP BY encounters.PATIENT;
'''

# Execute the query
preventive_visits = pandas.read_sql_query(query, rdbms)
preventive_visits.head()


,patient_id,last_preventive
0,0020dcbe-9f8d-920d-c008-d68debcef322,1941-10-13T03:05:01Z
1,0059312d-6c30-10fc-9c05-d631ce28cb17,1916-10-08T02:55:08Z
2,0060bded-e53c-348d-a1ab-fda3b7260151,1959-01-18T11:29:37Z
3,0074596f-5fd0-7965-db0f-cce71c81567d,2011-09-16T00:02:50Z
4,00875442-ebdf-6db2-9979-a09efa3a3c30,2020-03-31T13:48:44Z


## Merge Data and Identify Patients Overdue for Preventive Visits

We'll merge the two datasets and identify patients who either have no preventive visit recorded or whose last visit was more than a year ago.


In [14]:
# add preventive visits to patients_over_50 table
merged = patients_over_50.merge(preventive_visits, on='patient_id', how='left')

# overwrite column last_preventive in merged as the datetime value of whatever was there to begin with
merged['last_preventive'] = pandas.to_datetime(merged['last_preventive'], errors='coerce')

# calculate a cutoff date ocurring 12 months ago
cutoff = pandas.Timestamp.now(tz='UTC') - pandas.DateOffset(months=12)

# filter to rows where last_preventive is missing or ocurred over a year ago
merged_filtered = merged[(merged['last_preventive'].isna()) | (merged['last_preventive'] < cutoff)]

# sort the result by expense, high-to-low and show first 10 rows
merged_filtered_sorted = merged_filtered.sort_values(by='HEALTHCARE_EXPENSES', ascending=False)
merged_filtered_sorted[['patient_id', 'age', 'HEALTHCARE_EXPENSES', 'last_preventive']].head(10)
top_patients = merged_filtered_sorted[['patient_id', 'age', 'HEALTHCARE_EXPENSES', 'last_preventive']].head(10)
top_patients

,patient_id,age,HEALTHCARE_EXPENSES,last_preventive
516,dc18af25-7074-a277-def9-5806e7a48878,96.557937,2014822.78,2021-03-07 14:25:06+00:00
61,82ecedc1-945e-f6c1-f924-e7f8d3166729,79.260196,1979925.01,2021-01-21 10:43:57+00:00
1614,224db58f-e065-6c3e-96f6-c183a8f2d196,90.156842,1966703.52,2021-03-14 18:00:28+00:00
511,4835e469-a4df-7a77-53b0-7a1a408c50fb,93.705097,1943108.43,1991-11-30 17:32:52+00:00
695,19ac1829-945a-c4a3-63fb-ce50c89977ca,92.415569,1937726.20,2021-02-08 19:50:50+00:00
1162,4e7e5bf8-2823-8ec1-fe37-eba9c9d69463,95.572311,1910124.58,2021-07-14 11:24:08+00:00
981,a8acc8f0-9730-8483-32ec-f335166cee58,94.343016,1906655.70,2021-03-25 19:22:11+00:00
677,aefe9716-6072-791e-c0f7-8bb2d1227915,91.394351,1889216.34,2012-06-20 06:20:35+00:00
604,1ec3dd8f-29b0-cbfe-ab06-fcc154553d8a,99.416254,1865933.44,1996-01-27 06:05:40+00:00
340,80e1ab6a-c294-4fb2-65e0-c7b2a8b7c3c4,93.921387,1851301.18,2020-12-24 23:04:58+00:00


In [16]:
# --- Step 6: Describe ---

# Calculate the mean expense across the rows in top_patients
print(top_patients[['HEALTHCARE_EXPENSES']].mean())

HEALTHCARE_EXPENSES    1926551.718
dtype: float64


In [18]:
# --- Step 7: Save the output ---
top_patients.to_csv("top_patients.csv", index=False)
print("Output saved as top_patients.csv")


Output saved as top_patients.csv
